[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_30/2_sql.ipynb)

# Day 30: SQL final check

**Light day: a short final check (about 20 minutes), then rest.**

- Three mixed questions, no timer pressure. If one feels slow, that is fine: note it, do not grind.
- Then redo the 2 or 3 SQL items from your mistake log that you missed most often (the handbook Plan tab lists them).
- Finish with the checklist at the bottom of this notebook. Sleep matters more than one more query.

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (18 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Top customer in each country, ties included | hard | 5 |
| Q2 | Find the bugs | hard | 6 |
| Q3 | Whose fans are they? (review) | hard | 7 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Tables

**Real tables: Chinook** (a digital music store, 2009 to 2013; MIT license, lerocha/chinook-database):

| Table | One row is | Key columns |
|---|---|---|
| `Customer` | one customer (59) | `CustomerId`, `FirstName`, `LastName`, `Company` (often NULL), `City`, `State` (often NULL), `Country`, `Email`, `SupportRepId` |
| `Employee` | one employee (8) | `EmployeeId`, `FirstName`, `LastName`, `Title`, `ReportsTo` (manager id, NULL for the boss), `HireDate`, `City` |
| `Invoice` | one purchase (412) | `InvoiceId`, `CustomerId`, `InvoiceDate`, `BillingCity`, `BillingCountry`, `Total` |
| `InvoiceLine` | one track on an invoice (2,240) | `InvoiceLineId`, `InvoiceId`, `TrackId`, `UnitPrice`, `Quantity` |
| `Track` | one song or video (3,503) | `TrackId`, `Name`, `AlbumId`, `MediaTypeId`, `GenreId`, `Composer` (can be NULL), `Milliseconds`, `Bytes`, `UnitPrice` |
| `Album` / `Artist` | one album (347) / artist (275) | `AlbumId`, `Title`, `ArtistId` / `ArtistId`, `Name` |
| `Genre` / `MediaType` | one genre (25) / format (5) | `GenreId`, `Name` / `MediaTypeId`, `Name` |
| `Playlist` / `PlaylistTrack` | one playlist (18) / one track in a playlist (8,715) | `PlaylistId`, `Name` / `PlaylistId`, `TrackId` |

**Made-up short-video tables** (invented for practice, fixed seed; May 2024):

| Table | One row is | Columns |
|---|---|---|
| `tt_creators` | one creator (40) | `creator_id`, `category` (main category), `country` |
| `tt_videos` | one posted video | `video_id`, `creator_id`, `posted_at` ('YYYY-MM-DD HH:MM:SS'), `duration_s`, `category` |
| `tt_views` | one play of a video | `view_id`, `viewer_id`, `video_id`, `view_time`, `watch_s` (seconds watched; can be MORE than `duration_s` when the video loops), `source` (fyp / following) |
| `tt_follows` | one follow | `follower_id`, `creator_id`, `followed_at` |

The setup cell prints every table with its columns. Call `tables()` again any time.

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Real data: the Chinook digital music store (copied into the in-memory db).
_src = sqlite3.connect(data_path("chinook.sqlite", "https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_Sqlite.sqlite"))
_src.backup(db)
_src.close()

# Made-up short-video tables (generated with a fixed seed; numpy's legacy RandomState is frozen across versions).
import numpy as np

def make_video_tables(con, seed=25):
    rs = np.random.RandomState(seed)
    cats = ["dance", "food", "comedy", "education", "gaming"]
    nc = 40
    creators = pd.DataFrame({
        "creator_id": np.arange(1, nc + 1),
        "category": [cats[i] for i in rs.randint(0, 5, nc)],
        "country": [["US", "KR", "BR", "ID", "DE"][i] for i in rs.choice(5, nc, p=[.3, .25, .2, .15, .1])],
    })
    pop = rs.lognormal(0, 1.0, nc)                              # creator popularity
    start = pd.Timestamp("2024-05-01")
    vids = []
    vid = 1000
    for c in range(nc):
        for _ in range(1 + rs.poisson(6)):
            vid += 1
            posted = start + pd.Timedelta(seconds=int(rs.randint(0, 27 * 86400)))
            cat = creators.category[c] if rs.rand() < 0.85 else cats[rs.randint(0, 5)]
            vids.append((vid, c + 1, posted, int(rs.choice([15, 30, 45, 60, 90, 180])), cat))
    videos = pd.DataFrame(vids, columns=["video_id", "creator_id", "posted_at", "duration_s", "category"])
    videos = videos.sort_values("posted_at", kind="mergesort").reset_index(drop=True)
    vq = rs.beta(2, 2, len(videos))                             # video quality: how much of it people watch
    w = pop[videos.creator_id.values - 1] * (0.5 + vq)
    posted_s = ((videos.posted_at - start).dt.total_seconds()).values
    views, follows, followed = [], [], set()
    for u in range(1, 1201):
        n_sess = rs.poisson(4)
        for _ in range(n_sess):
            t = int(rs.randint(1 * 86400, 31 * 86400))
            n_vid = 1 + rs.geometric(1 / 6.0)
            source = "following" if rs.rand() < 0.15 else "fyp"
            for _ in range(n_vid):
                ok = np.nonzero(posted_s < t)[0]
                if len(ok) == 0:
                    break
                p = w[ok] / w[ok].sum()
                i = ok[rs.choice(len(ok), p=p)]
                dur = videos.duration_s[i]
                frac = min(rs.beta(1 + 4 * vq[i], 2), 1.0)
                if rs.rand() < 0.08:
                    frac += 1 + rs.rand()                       # rewatch loop: watched more than the length
                watch = max(1, int(round(frac * dur)))
                views.append((u, int(videos.video_id[i]), start + pd.Timedelta(seconds=t), watch, source))
                cid = int(videos.creator_id[i])
                if frac > 0.8 and (u, cid) not in followed and rs.rand() < 0.06:
                    followed.add((u, cid))
                    follows.append((u, cid, start + pd.Timedelta(seconds=t + watch + 2)))
                t += watch + int(rs.randint(1, 15))
                source = "fyp"
    fmt = "%Y-%m-%d %H:%M:%S"
    views = pd.DataFrame(views, columns=["viewer_id", "video_id", "view_time", "watch_s", "source"])
    views = views.sort_values(["view_time", "viewer_id"], kind="mergesort").reset_index(drop=True)
    views.insert(0, "view_id", np.arange(1, len(views) + 1))
    views["view_time"] = views["view_time"].dt.strftime(fmt)
    follows = pd.DataFrame(follows, columns=["follower_id", "creator_id", "followed_at"])
    follows = follows.sort_values("followed_at", kind="mergesort").reset_index(drop=True)
    follows["followed_at"] = follows["followed_at"].dt.strftime(fmt)
    videos["posted_at"] = videos["posted_at"].dt.strftime(fmt)
    for name, df in [("tt_creators", creators), ("tt_videos", videos), ("tt_views", views), ("tt_follows", follows)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_video_tables(db)

def tables():
    """Prints every table with its row count and columns."""
    for (t,) in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
        cols = [r[1] for r in db.execute(f'PRAGMA table_info("{t}")')]
        n = db.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
        print(f"{t} ({n:,} rows): {', '.join(cols)}")

tables()

---
### Q1. Top customer in each country, ties included

*hard, about 5 min*

Chinook (real). For countries with at least 3 customers, return the customer(s) with the highest total
spend (`SUM(Invoice.Total)`): `Country`, `customer` (first + last name), `total` (2 decimals). If several customers
tie for first place, show all of them. Order by total descending.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'highest per country' and 'show all ties'. Pattern: top-N per group with RANK (not ROW_NUMBER), plus a COUNT window for the group size filter.

</details>

<details><summary><b>Hint 2</b></summary>

1. `spend`: SUM(Total) per country and customer.
2. `RANK() OVER (PARTITION BY Country ORDER BY total DESC)` and `COUNT(*) OVER (PARTITION BY Country)`.
3. Keep rank 1 and count >= 3.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH spend AS (
  SELECT c.Country, c.CustomerId, c.FirstName || ' ' || c.LastName AS customer,
         SUM(i.Total) AS total
  FROM Customer c
  JOIN Invoice i ON i.CustomerId = c.CustomerId
  GROUP BY c.Country, c.CustomerId, c.FirstName, c.LastName
), r AS (
  SELECT *,
    RANK() OVER (PARTITION BY Country ORDER BY total DESC) AS rk,
    COUNT(*) OVER (PARTITION BY Country) AS n_customers
  FROM spend
)
SELECT Country, customer, ROUND(total, 2) AS total
FROM r
WHERE rk = 1 AND n_customers >= 3
ORDER BY total DESC
```

Result:

| Country | customer | total |
|---|---|---|
| USA | Richard Cunningham | 47.62 |
| Germany | Fynn Zimmermann | 43.62 |
| France | Isabelle Mercier | 40.62 |
| Brazil | Luís Gonçalves | 39.62 |
| Canada | François Tremblay | 39.62 |
| United Kingdom | Emma Jones | 37.62 |
| United Kingdom | Phil Hughes | 37.62 |
| United Kingdom | Steve Murray | 37.62 |

**Why:** RANK gives every tied customer rank 1, so the United Kingdom returns three customers at 37.62; ROW_NUMBER would silently pick one. The group-size filter is a window COUNT, computed before the rank filter, so it counts all customers of the country, not just the winners.

**Complexity:** One join, one aggregation, two windows over 59 customers.

**Common mistakes:** ROW_NUMBER (drops ties). `WHERE n_customers >= 3` inside the same SELECT that defines it (not allowed: filter in the outer query). Grouping by name only (two customers with the same name merge).

**Learn more:** [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n), [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking)

</details>

---
### Q2. Find the bugs

*hard, about 6 min*

A teammate wrote this query for "conversion rate per country = users with a completed order / all users",
but the numbers look too high. Find **three** bugs and say how to fix each one.

```sql
SELECT u.country,
  COUNT(*) AS users,
  COUNT(p.order_id) / COUNT(*) * 100 AS conversion_pct
FROM users u
LEFT JOIN orders p ON p.user_id = u.user_id
WHERE p.status = 'completed'
GROUP BY u.country;
```

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: a rate built from a LEFT JOIN. Check three things: where the filter sits, what COUNT counts after the join, and the data type of the division.

</details>

<details><summary><b>Hint 2</b></summary>

1. What does `WHERE p.status = ...` do to users without orders?
2. A user with 3 orders: how many rows?
3. What is `7 / 10` in integer arithmetic?

</details>

<details><summary><b>Solution</b></summary>

**Bug 1: the filter is in WHERE.** For users without orders `p.status` is NULL, so WHERE removes them and
the LEFT JOIN becomes an inner join: only buyers remain, and the rate is close to 100%. Fix: move it into the join,
`LEFT JOIN orders p ON p.user_id = u.user_id AND p.status = 'completed'`.

**Bug 2: rows are orders, not users.** A user with 3 orders gives 3 rows, so `COUNT(*)` counts orders and
`COUNT(p.order_id)` counts orders too. Fix: `COUNT(DISTINCT u.user_id)` for users and
`COUNT(DISTINCT p.user_id)` for converted users.

**Bug 3: integer division.** In PostgreSQL and SQLite, `7 / 10` is 0, so the rate is 0 or 100 (and multiplying by 100
after dividing is too late). Fix: `100.0 * COUNT(DISTINCT p.user_id) / COUNT(DISTINCT u.user_id)`.

Fixed query:

```sql
SELECT u.country,
  COUNT(DISTINCT u.user_id) AS users,
  ROUND(100.0 * COUNT(DISTINCT p.user_id) / COUNT(DISTINCT u.user_id), 1) AS conversion_pct
FROM users u
LEFT JOIN orders p ON p.user_id = u.user_id AND p.status = 'completed'
GROUP BY u.country;
```

Bonus: users with a NULL country form their own group; say whether you keep it or label it 'unknown' with COALESCE.

**Why:** These three bugs (WHERE after LEFT JOIN, fan-out after a join, integer division) cause most wrong ratios in SQL exams. Checking for them is the 30 second review to do before you say 'done'.

**Learn more:** [sql-left-join-nulls](https://mahsa-agz.github.io/ds-handbook/#sql-left-join-nulls), [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios), [sql-exam-approach](https://mahsa-agz.github.io/ds-handbook/#sql-exam-approach)

</details>

---
### Q3. Whose fans are they? (review)

*hard, about 7 min*

Short-video app (made up). A viewer's **favourite creator** is the creator whose videos they watched for the
most seconds in total (ties: the lower `creator_id`). Return the 5 creators who are the favourite of the most viewers:
`creator_id`, `fans`, `pct` (% of all viewers, 1 decimal). Order by fans descending, then creator_id.

*Follow-up:* one creator is the favourite of a third of all viewers. Is that a problem for the platform?

*Source: [DataLemur TikTok SQL questions (inspiration)](https://datalemur.com/blog/tiktok-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'the top one per viewer', then count those. Pattern: top-N per group (ROW_NUMBER with a tiebreaker), then an aggregation over the winners.

</details>

<details><summary><b>Hint 2</b></summary>

1. `w`: SUM(watch_s) per viewer and creator (views join videos).
2. `ROW_NUMBER() OVER (PARTITION BY viewer_id ORDER BY secs DESC, creator_id)`, keep 1.
3. COUNT per creator; divide by `(SELECT COUNT(DISTINCT viewer_id) FROM tt_views)`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH w AS (
  SELECT x.viewer_id, v.creator_id, SUM(x.watch_s) AS secs
  FROM tt_views x
  JOIN tt_videos v ON v.video_id = x.video_id
  GROUP BY x.viewer_id, v.creator_id
), top AS (
  SELECT viewer_id, creator_id,
    ROW_NUMBER() OVER (PARTITION BY viewer_id ORDER BY secs DESC, creator_id) AS rn
  FROM w
)
SELECT creator_id, COUNT(*) AS fans,
  ROUND(100.0 * COUNT(*) / (SELECT COUNT(DISTINCT viewer_id) FROM tt_views), 1) AS pct
FROM top
WHERE rn = 1
GROUP BY creator_id
ORDER BY fans DESC, creator_id
LIMIT 5
```

Result:

| creator_id | fans | pct |
|---|---|---|
| 16 | 401 | 33.9 |
| 38 | 107 | 9.0 |
| 25 | 64 | 5.4 |
| 5 | 62 | 5.2 |
| 23 | 53 | 4.5 |

**Why:** Here ROW_NUMBER is right (the task defines a tiebreaker and wants exactly one favourite per viewer). Creator 16 is the favourite of 401 viewers (33.9%). Follow-up: heavy concentration means the feed may over-expose one creator (a popularity feedback loop); new creators get little exposure and viewers may tire. Look at diversity metrics (share of watch time from the top creator per viewer) and at exploration in ranking.

**Complexity:** One join and aggregation, one window over (viewer, creator) pairs.

**Common mistakes:** Favourite by number of views instead of seconds (the task says seconds). RANK without a tiebreaker (a viewer can then have two favourites and is counted twice).

**Learn more:** [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n), [sql-aggregation](https://mahsa-agz.github.io/ds-handbook/#sql-aggregation)

</details>

---
## Checklist for the day before the exam

**SQL, 10 minutes of recall (say it, do not code it):**
- Query order: FROM, JOIN, WHERE, GROUP BY, HAVING, window functions, SELECT, DISTINCT, ORDER BY, LIMIT.
- The 6 shapes you met most: top-N per group (ROW_NUMBER / RANK), running totals (SUM OVER, ROWS frame),
  LAG/LEAD comparisons, gaps and islands (date minus ROW_NUMBER), retention / funnels (LEFT JOIN + CASE flags),
  median / percentiles (ROW_NUMBER and COUNT, or CUME_DIST).
- The 3 bugs from Q2: filter of a LEFT JOIN in WHERE, fan-out after a join, integer division.
- Dialect notes: PostgreSQL has `DATE_TRUNC`, `INTERVAL`, `FILTER (WHERE ...)`, `PERCENTILE_CONT`; SQLite does not.

**In the exam:**
- Restate the question, ask about the grain (one row = ?), duplicates, NULLs and time zone.
- Say the plan in 2 sentences, write CTEs with clear names, run a sanity check (row counts, a known user).
- When done, offer one follow-up yourself: "if the table had duplicates I would ...", "at scale I would ...".

**Logistics:**
- Test the editor or shared doc link, camera, microphone; have water and paper ready.
- Prepare 2 questions for the examiner about the team's data and metrics.
- Stop studying early in the evening. Sleep is part of the preparation.

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_30/3_stats.ipynb)